# 01 · Attention from scratch

> **Paper:** Appendix A.1 "Preliminaries: Multi-head attention layers", Eq. (3)–(8)

DETR is *"DEtection TRansformer"*. If the transformer half is a black box, the rest of this tutorial will feel like memorizing, not understanding. So before we touch DETR, we build attention **from scratch, by hand**, and check our version against PyTorch's numerically.

**By the end you'll be able to answer:**
- What are Q, K, V, and why three of them?
- Why divide by √d?
- What's the difference between *self*-attention and *cross*-attention? (DETR's decoder uses both)
- Why does a transformer need positional encodings at all?
- Why must DETR's 100 object queries be *different from each other*?

Those last two questions are the whole reason notebooks `03` and `04` look the way they do.

**Nothing here needs a GPU or a download.** Small tensors, printed in full, so you can follow every number.

> **New to PyTorch itself?** This notebook assumes you can read `x.permute(2, 0, 1)` without flinching. If you can't yet, spend twenty minutes in [`00 · PyTorch essentials`](00_pytorch_essentials.ipynb) first — §2 (shape surgery) and §6 (matmul and the shape of attention) are the direct prerequisites for what follows.

In [1]:
%matplotlib inline
import torch, math
import torch.nn.functional as F
from torch import nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
torch.set_printoptions(precision=3, sci_mode=False, linewidth=100)
print("torch", torch.__version__)

torch 2.14.0


## 1. The one-sentence intuition

> Attention is a **soft dictionary lookup.**

An ordinary Python dict is a *hard* lookup: `d["cat"]` returns exactly one value, and a key either matches or it doesn't.

Attention is the soft version:
- Every item offers a **key** (what it is) and a **value** (what it carries).
- You arrive with a **query** (what you're looking for).
- Instead of picking one key, you measure similarity against *all* keys, turn those similarities into weights that sum to 1, and return the **weighted blend of all values**.

| Python dict | attention |
|---|---|
| `q` — the key you look up *with* | **Q** — what this position is asking for |
| `key` — what each entry is filed under | **K** — what each position advertises |
| `value` — what each entry stores | **V** — what each position hands over |
| `d[q]` — the one value you get back | **the output** — a weighted blend of every value |
| exact match, or `KeyError` | dot-product similarity → softmax over *all* keys |

Note that `d[q]` lines up with the **output**, not with Q. Q is what you go in with; `d[q]` is what you come out with. Attention's output is the soft answer to the lookup.

### How much of this is the paper's, and how much is a teaching device?

Worth separating, because the analogy is doing real work below and you should know which parts you can cite.

**The paper's**, not ours:

- The three names, and the key-value framing. Vaswani et al. §3.2 opens: *"An attention function can be described as mapping a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors."* The pairing of keys with values is in the definition itself.
- DETR keeps the same vocabulary: Appendix A.1 calls its inputs the *"query sequence"* `X_q` and the *"key-value sequence"* `X_kv`.
- The weighted blend. DETR A.1, right after Eq. (8): *"The final output is the aggregation of values weighted by attention weights."* Vaswani §3.2: *"The output is computed as a weighted sum of the values, where the weight assigned to each value is computed by a compatibility function of the query with the corresponding key."*

**Ours** — a gloss on top:

- The word **dictionary**, and the `d[q]` column. Neither paper mentions dictionaries, databases, or retrieval, and **neither explains where the names came from**. The lookup story is a standard teaching device (you will meet it in most courses), not something you can cite.
- "Hard vs soft" as a framing.

So: the vocabulary and the mechanism are quoted; the *story explaining why the vocabulary was chosen* is reconstruction. It is a good reconstruction — the names would be a strange coincidence otherwise — but treat it as a mnemonic, and trust the equations in §2 over it wherever the two seem to disagree.

In DETR's decoder the analogy maps cleanly: an object query asks *"is there an object with these properties?"* and gets back a blend of image features, weighted by how well each image patch answers.

## 2. A worked example with 5 tokens

We'll use a **sequence of 5 tokens, each 8-dimensional**, in a **batch of one**. Small enough to print, big enough to be real.

**Everything in this section is self-attention.** Q, K and V will all be projections of the same `x`, so the 5 tokens are looking at each other. That is why the score matrix comes out square, `(5, 5)`. Section 3 changes exactly one thing — where Q comes from — and that one change is the whole difference between self- and cross-attention. Watch for the moment the matrix stops being square.

**Three widths, deliberately all different.** Most tutorials set every width equal, which hides the shape rules behind a coincidence. We won't:

| | symbol | here | what it is |
|---|---|---|---|
| batch | `B` | 1 | how many sequences travel together |
| sequence length | `L` | 5 | how many tokens in each |
| input width | `D_in` | 8 | how wide each incoming token is |
| query/key width | `D_k` | 4 | the space queries and keys are compared in |
| value width | `D_v` | 6 | how wide the thing being retrieved is |

Only one of those is forced: **Q and K must share a width**, because they meet in a dot product. `D_v` is free, and `D_in` need not equal either. So the output of this section comes out `(1, 5, 6)` — *not* the `(1, 5, 8)` we started with. That is normal, and §3 explains why real transformers usually hide it.

**Keep the names apart even once the numbers agree.** In practice almost every transformer sets `D_in == D_k == D_v == d_model` — DETR uses 256 for all three. That does not make them one thing; it makes three roles that happen to share a number. Code that calls them all `D` cannot tell you *which role* an axis was playing when a shape goes wrong, and "expected 256, got 256" is not a debugging message anyone enjoys. Separate names cost nothing and turn a shape bug into a sentence you can read. We give them different values here only so you can watch which one ends up where.

**We keep the batch axis the whole way through.** Every tensor below is `(B, L, …)`, even though `B = 1`. It costs one extra bracket in the printouts and buys something worth more: none of the code below silently depends on there being exactly one sequence, and the shapes you read here are the shapes you would see in a real training loop. It also keeps one trap visible — `k.T` reverses *all* axes and quietly breaks on a 3-D tensor, which is why every transpose of an **activation** here is written `k.transpose(-2, -1)`. (`.T` on a 2-D *weight* matrix, as in `W.T`, stays perfectly fine — that is the one place you will still see it below.)

`B`, `L` and `D` are the axis letters from [`00 · PyTorch essentials`](00_pytorch_essentials.ipynb). One warning: PyTorch's transformer modules call the feature width `E` (for *embedding*), and DETR calls it `d_model`. Same idea, three names.

In [2]:
B, L = 1, 5                # batch size, sequence length (5 tokens)
D_in, D_k, D_v = 8, 4, 6   # input / query-key / value widths -- all different on purpose

x = torch.randn(B, L, D_in)

print("x:", tuple(x.shape), "= (batch, tokens, features)")
print(x)

x — our input sequence
shape: (1, 5, 8)  = (batch, tokens, features)

We keep the batch axis for the rest of the notebook, so every shape below
is what you would actually see with B > 1. Nothing here assumes B == 1.
tensor([[[-1.126, -1.152, -0.251, -0.434,  0.849,  0.692, -0.316, -2.115],
         [ 0.322, -1.263,  0.350,  0.308,  0.120,  1.238,  1.117, -0.247],
         [-1.353, -1.696,  0.567,  0.794,  0.599, -1.555, -0.341,  1.853],
         [-0.216, -0.743,  0.563,  0.260, -0.174, -0.679,  0.938,  0.489],
         [ 1.203,  0.085, -1.200, -0.005, -0.518, -0.307, -1.581,  1.707]]])


### Step 1 — project into Q, K, V

Q, K and V are all **linear projections of the same input `x`** — that sameness is what makes this *self*-attention. Three different learned matrices, so the same token can *ask* one thing, *advertise* another, and *carry* a third.

Notice the shapes of the three matrices. `W_q` and `W_k` both map `D_in → D_k`, because their outputs have to meet in a dot product. `W_v` maps `D_in → D_v`, and nothing constrains `D_v` at all — V is only ever *summed*, never multiplied against Q or K.

`nn.Linear` maps the **last** axis only and treats `B` and `L` as batch, so one call projects every token of every sequence at once. That is the "one weight matrix applied independently to each row" idea from notebook `03` — `nn.Linear` over a sequence.

In [3]:
W_q = nn.Linear(D_in, D_k, bias=False)    # D_in -> D_k
W_k = nn.Linear(D_in, D_k, bias=False)    # D_in -> D_k, forced to match W_q
W_v = nn.Linear(D_in, D_v, bias=False)    # D_in -> D_v, nothing forces this one

q = W_q(x)      # "what am I looking for?"
k = W_k(x)      # "what do I advertise?"
v = W_v(x)      # "what do I hand over?"

for name, t in [("x", x), ("q", q), ("k", k), ("v", v)]:
    print(f"  {name}  {tuple(t.shape)}")

print("\nsame token, three different projections:")
print("  q[0, 0] =", q[0, 0])
print("  k[0, 0] =", k[0, 0])

x: (1, 5, 8) -- the same 5 tokens feed all three projections
q: (1, 5, 4)  k: (1, 5, 4)  v: (1, 5, 6)

q and k are 4 wide because they meet in a dot product.
v is 6 wide because nothing says otherwise.
B and L are untouched -- Linear only ever rewrites the last axis.

q[0, 0] and k[0, 0] both come from token 0 of sequence 0, but differ:
  q[0, 0] = tensor([ 0.457,  0.575, -0.402,  0.470], grad_fn=<SelectBackward0>)
  k[0, 0] = tensor([ 0.880,  0.557,  0.194, -0.316], grad_fn=<SelectBackward0>)


### Step 2 — similarity scores: every query against every key

`scores[i, j]` = how much **token i's query** matches **token j's key**. A dot product: large when the two vectors point the same way.

In [4]:
# .transpose(-2, -1), not .T -- on a 3-D tensor .T reverses ALL axes, giving (D_k, L, B).
scores = q @ k.transpose(-2, -1)   # (B, L, D_k) @ (B, D_k, L) -> (B, L, L)

print("scores:", tuple(scores.shape), "= one score per (query, key) pair")
print(scores)

# D_k was summed away by the dot product -- that is why q and k had to share it,
# and why the result does not depend on D_v at all.
print("\nscores[0, 0, 2], token 0 asking about token 2:", scores[0, 0, 2].item())
print("  the same dot product by hand              :", torch.dot(q[0, 0], k[0, 2]).item())

scores shape: (1, 5, 5)  = (B, L, L), one score per (query, key) pair
   .transpose(-2, -1), not .T: on a 3-D tensor .T reverses ALL axes
   and would give (D_k, L, B). See notebook 00 section 6.
tensor([[[ 0.496,  0.083, -0.859, -0.492, -0.177],
         [ 0.436,  0.477, -0.308, -0.174,  0.180],
         [-0.076,  0.299,  0.675,  0.437, -0.195],
         [-0.459, -0.028,  0.657,  0.471, -0.071],
         [ 0.939,  0.819, -0.161, -0.238,  0.167]]], grad_fn=<UnsafeViewBackward0>)

Note the D_k axis has vanished -- it was summed over by the dot product.
That is why q and k had to share it, and why the result does not depend on D_v.

row i = token i's query scored against ALL keys
scores[0, 0, 2] is token 0 asking about token 2: -0.8585454821586609
  verify by hand: -0.8585454821586609


### Step 3 — scale by √d

Divide by `√D_k` — the width Q and K were compared in, not the input width and not the value width. Here's *why*, demonstrated rather than asserted: the dot product of two random `d`-dimensional vectors has variance that **grows with `d`**. Large scores make softmax saturate — one weight goes to ~1.0, the rest to ~0 — and a saturated softmax has almost **zero gradient**, so the layer stops learning.

In [5]:
print("std of raw dot products as the compared width d grows:")
for d in [8, 64, 256, 1024]:
    a, b = torch.randn(512, d), torch.randn(512, d)
    # a @ b.T is exactly the score matrix from Step 2: every row of a scored
    # against every row of b. Each of its 512*512 entries is one dot product,
    # so the whole matrix is a large sample to take the std of.
    # (a and b are 2-D here, so .T is safe -- see the note in section 2.)
    raw = a @ b.T
    print(f"   d={d:5d}:  std(q·k) = {raw.std():7.2f}   after /sqrt(d) = {(raw/math.sqrt(d)).std():.2f}")

print("\nEffect on softmax (one row of scores, scaled up vs down):")
row = torch.tensor([2.0, 1.0, 0.5, 0.2])
for mult, label in [(1, "well-scaled"), (10, "unscaled (large d)")]:
    w = (row * mult).softmax(-1)
    print(f"   {label:<20} softmax = {w}   max={w.max():.3f}")
print("\n-> unscaled: one weight ~1.0, gradient vanishes. That is what /sqrt(d) prevents.")

std of raw dot products as the compared width d grows:
   d=    8:  std(q·k) =    2.80   after /sqrt(d) = 0.99
   d=   64:  std(q·k) =    8.03   after /sqrt(d) = 1.00
   d=  256:  std(q·k) =   16.04   after /sqrt(d) = 1.00
   d= 1024:  std(q·k) =   31.99   after /sqrt(d) = 1.00

Effect on softmax (one row of scores, scaled up vs down):
   well-scaled          softmax = tensor([0.569, 0.209, 0.127, 0.094])   max=0.569
   unscaled (large d)   softmax = tensor([1.000, 0.000, 0.000, 0.000])   max=1.000

-> unscaled: one weight ~1.0, gradient vanishes. That is what /sqrt(d) prevents.


In [6]:
scaled = scores / math.sqrt(D_k)   # D_k = 4, so this divides by 2

print("scaled scores:", tuple(scaled.shape))
print(scaled)

scaled scores:
Shape:  torch.Size([1, 5, 5])
tensor([[[ 0.248,  0.042, -0.429, -0.246, -0.089],
         [ 0.218,  0.238, -0.154, -0.087,  0.090],
         [-0.038,  0.149,  0.338,  0.219, -0.098],
         [-0.230, -0.014,  0.328,  0.236, -0.035],
         [ 0.470,  0.410, -0.081, -0.119,  0.084]]], grad_fn=<DivBackward0>)


### Step 4 — softmax into attention weights

Row-wise softmax. Each row becomes a **probability distribution over the `L` tokens**: "how much of my output should come from each one?"

In [7]:
attn = scaled.softmax(dim=-1)      # over the LAST axis: the keys

print("attention weights:", tuple(attn.shape))
print(attn)

# sum(-1) and sum(-2), not sum(0): with a batch axis, axis 0 is B.
print("\nrows sum to 1 -- each is a distribution over keys:", attn.sum(-1))
print("columns do not, and that trips people up        :", attn.sum(-2))

attention weights: (1, 5, 5)
tensor([[[0.274, 0.223, 0.139, 0.167, 0.196],
         [0.231, 0.236, 0.159, 0.170, 0.203],
         [0.170, 0.205, 0.247, 0.219, 0.160],
         [0.147, 0.183, 0.257, 0.234, 0.179],
         [0.266, 0.251, 0.154, 0.148, 0.181]]], grad_fn=<SoftmaxBackward0>)

every ROW sums to 1 (it is a distribution over tokens):
  row sums   attn.sum(-1): tensor([[1.000, 1.000, 1.000, 1.000, 1.000]], grad_fn=<SumBackward1>)

columns do NOT sum to 1 -- a common confusion:
  column sums attn.sum(-2): tensor([[1.089, 1.097, 0.956, 0.939, 0.919]], grad_fn=<SumBackward1>)

Note -2, not 0. With a batch axis, sum(0) would collapse B instead of the
query axis -- exactly the kind of bug keeping B around makes visible.


### Step 5 — weighted sum of values

Finally, blend the values using those weights.

In [8]:
out = attn @ v            # (B, L, L) @ (B, L, D_v) -> (B, L, D_v)
print("input  x  :", tuple(x.shape), " = (B, tokens, D_in)")
print("output out:", tuple(out.shape), " = (B, tokens, D_v)   <- DIFFERENT width, and that is fine")
print()
print("The output is as wide as V, because V is the only thing being blended.")
print("Q's width vanished at the dot product; D_in vanished at the projections.")
print(out)
print()
print("token 0's output is a weighted blend of ALL", L, "value vectors:")
manual = sum(attn[0, 0, j] * v[0, j] for j in range(L))
print("  by hand :", manual)
print("  matmul  :", out[0, 0])
print("  equal?  ", torch.allclose(manual, out[0, 0], atol=1e-6))


input  x  : (1, 5, 8)  = (B, tokens, D_in)
output out: (1, 5, 6)  = (B, tokens, D_v)   <- DIFFERENT width, and that is fine

The output is as wide as V, because V is the only thing being blended.
Q's width vanished at the dot product; D_in vanished at the projections.
tensor([[[-0.116,  0.251,  0.120,  0.160, -0.188,  0.066],
         [-0.114,  0.254,  0.111,  0.126, -0.214,  0.064],
         [-0.142,  0.300,  0.111,  0.175, -0.202,  0.025],
         [-0.124,  0.286,  0.072,  0.165, -0.204,  0.042],
         [-0.135,  0.268,  0.150,  0.156, -0.202,  0.049]]], grad_fn=<UnsafeViewBackward0>)

token 0's output is a weighted blend of ALL 5 value vectors:
  by hand : tensor([-0.116,  0.251,  0.120,  0.160, -0.188,  0.066], grad_fn=<AddBackward0>)
  matmul  : tensor([-0.116,  0.251,  0.120,  0.160, -0.188,  0.066], grad_fn=<SelectBackward0>)
  equal?   True


### The whole thing in one line

That's the entire attention mechanism:

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{D_k}}\right)V$$

**A note on where that formula comes from.** This one-line form is Vaswani et al.'s, DETR's reference [47]. The DETR paper itself splits the same computation across two statements in Appendix A.1. **Eq. (8)** gives only the *weights*:

$$\alpha_{i,j} = \frac{1}{Z_i}\,e^{\frac{1}{\sqrt{d'}} Q_i^\top K_j}, \qquad Z_i = \sum_{j=1}^{N_{kv}} e^{\frac{1}{\sqrt{d'}} Q_i^\top K_j}$$

and the sentence immediately after it — **unnumbered** — applies them to the values: $\text{attn}_i(X_q, X_{kv}, T') = \sum_{j} \alpha_{i,j} V_j$.

So Eq. (8) is our `attn` variable, not our `out`. Same computation, written in two steps rather than one. The paper's $d'$ is the **per-head** width $d/M$ — the thing §4 splits `D_k` into.


In [9]:
def attention(q, k, v):
    """Scaled dot-product attention. Returns (output, attention_weights)."""
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
    w = scores.softmax(-1)
    return w @ v, w

mine, w = attention(q, k, v)
theirs = F.scaled_dot_product_attention(q, k, v)     # PyTorch's built-in

print("my implementation matches PyTorch:", torch.allclose(mine, theirs, atol=1e-6))
print("max abs difference:", (mine - theirs).abs().max().item())

my implementation matches PyTorch: True
max abs difference: 2.9802322387695312e-08


## 3. Self-attention vs cross-attention

The single most useful distinction for reading DETR. The formula is **identical**; only *where Q, K, V come from* changes.

Section 2 was the self-attention case all along: one `x`, three projections of it. Here we finally feed Q from somewhere else.

| | Q from | K, V from | In DETR |
|---|---|---|---|
| **Self**-attention | the sequence itself | the same sequence | encoder: image patches ↔ image patches<br>decoder: query ↔ query (kills duplicates) |
| **Cross**-attention | sequence A | sequence **B** | decoder: object queries → image features |

Cross-attention is how DETR's 100 object queries actually *see* the image.

In [21]:
# SELF-attention: L image tokens attending to each other
img = torch.randn(B, 4, D_in)                    # (B, 4, D_in)
o_self, w_self = attention(W_q(img), W_k(img), W_v(img))
print("self : Q,K,V all from img(4)   -> attn", tuple(w_self.shape), " out", tuple(o_self.shape))

# CROSS-attention: 2 object queries reading from those 4 image tokens
queries = torch.randn(B, 2, D_in)                # (B, 2, D_in)
o_cross, w_cross = attention(W_q(queries), W_k(img), W_v(img))
print("cross: Q from queries(2), K,V from img(4)")
print("       -> attn", tuple(w_cross.shape), " out", tuple(o_cross.shape))
print()
print("KEY: output length follows the QUERY, not the keys.")
print("  2 queries in -> 2 outputs out, no matter how many image tokens there are.")
print("  That is exactly how DETR turns 850 image tokens into 100 detections.")


self : Q,K,V all from img(4)   -> attn (1, 4, 4)  out (1, 4, 6)
cross: Q from queries(2), K,V from img(4)
       -> attn (1, 2, 4)  out (1, 2, 6)

KEY: output length follows the QUERY, not the keys.
  2 queries in -> 2 outputs out, no matter how many image tokens there are.
  That is exactly how DETR turns 850 image tokens into 100 detections.


Re-read that last point — it is the structural reason DETR works. The decoder's output has one row **per query**, so N queries always produce exactly N predictions, regardless of image size. That's the "fixed set of N predictions" from the paper's §3.1.

### The shape rules, all of them

You have now seen both free axes in action, so here is the complete set of constraints. There are only two.

| | must match? | why |
|---|---|---|
| Q and K — feature width | **yes** | they meet in the dot product `q @ k.T` |
| K and V — sequence length | **yes** | the weights are `(L_q, L_k)` and get applied to V's rows |
| Q and V — feature width | no | never multiplied together — §2 used 4 and 6 |
| Q and K/V — sequence length | no | when they differ, that *is* cross-attention |
| **input and output width** | **no** | the output is as wide as V |

Everything else is a convention someone chose. Let's confirm the two real rules by breaking them:

In [30]:
print("the two rules, enforced by PyTorch:")
try:
    attention(torch.randn(B, 4, 8), torch.randn(B, 4, 16), torch.randn(B, 4, 8))
except RuntimeError as e:
    print("   q and k of different widths  ->", str(e).splitlines()[0][:60])
try:
    attention(torch.randn(B, 4, 8), torch.randn(B, 4, 8), torch.randn(B, 5, 8))
except RuntimeError as e:
    print("   k and v of different lengths ->", str(e).splitlines()[0][:60])

print("\nand everything the rules leave free:")
for (lq, dq), (lk, dk), (lv, dv) in [((4, 8), (4, 8), (4, 8)),
                                     ((2, 8), (4, 8), (4, 8)),
                                     ((4, 4), (4, 4), (4, 6)),
                                     ((2, 4), (9, 4), (9, 32))]:
    o, w = attention(torch.randn(B, lq, dq), torch.randn(B, lk, dk), torch.randn(B, lv, dv))
    print(f"   q(B,{lq},{dq}) k(B,{lk},{dk}) v(B,{lv},{dv})".ljust(40),
          f"-> attn {str(tuple(w.shape)):<12} out {tuple(o.shape)}")
print("\n   out length = q length,  out width = v width.  Always.")


the two rules, enforced by PyTorch:
   q and k of different widths  -> Expected size for first two dimensions of batch2 tensor to b
   k and v of different lengths -> Expected size for first two dimensions of batch2 tensor to b

and everything the rules leave free:
   q(B,4,8) k(B,4,8) v(B,4,8)            -> attn (1, 4, 4)    out (1, 4, 8)
   q(B,2,8) k(B,4,8) v(B,4,8)            -> attn (1, 2, 4)    out (1, 2, 8)
   q(B,4,4) k(B,4,4) v(B,4,6)            -> attn (1, 4, 4)    out (1, 4, 6)
   q(B,2,4) k(B,9,4) v(B,9,32)           -> attn (1, 2, 9)    out (1, 2, 32)

   out length = q length,  out width = v width.  Always.


**So why do real transformers make every width equal?** Because of what *wraps* attention, not attention itself.

A transformer layer puts a **residual connection** around every sublayer ([`transformer.py:157`](../models/transformer.py#L157)):

```python
src2 = self.self_attn(q, k, value=src, ...)[0]
src = src + self.dropout1(src2)        # <- this addition is the constraint
```

`x + f(x)` only works if `f(x)` has `x`'s shape. So every sublayer is *built* to return `d_model` wide, and `nn.MultiheadAttention` finishes with an `out_proj` that maps back to `embed_dim` however wide K and V were. DETR holds `d_model = 256` from the encoder's first layer to the decoder's last for exactly this reason.

But the freedom is still there, and DETR uses it a couple of lines further down the same file: the feed-forward sublayer goes **256 → 2048 → 256** ([`transformer.py:159`](../models/transformer.py#L159)). Unconstrained in the middle; it only has to come back before the residual.

In [12]:
# nn.MultiheadAttention exposes the freedom directly, via kdim / vdim.
# batch_first=True so it speaks our (B, L, E) convention -- see the note in section 4.
mha_wide = nn.MultiheadAttention(embed_dim=8, num_heads=2, kdim=16, vdim=32, batch_first=True)
Q, K, V = torch.randn(B, 3, 8), torch.randn(B, 5, 16), torch.randn(B, 5, 32)
o, _ = mha_wide(Q, K, V)

print("Q", tuple(Q.shape), " K", tuple(K.shape), " V", tuple(V.shape))
print("->", tuple(o.shape), " back to embed_dim=8, because of out_proj:",
      tuple(mha_wide.out_proj.weight.shape))
print()
print("So the module hides the freedom: inside, widths differ; at the door, embed_dim.")
print("That is what makes the residual connection x + attn(x) legal.")
print()

# and DETR's own feed-forward, where the width deliberately does NOT match
ff1, ff2 = nn.Linear(256, 2048), nn.Linear(2048, 256)
h = torch.randn(850, 1, 256)
print("DETR's FFN:", tuple(h.shape), "->", tuple(ff1(h).shape), "->", tuple(ff2(ff1(h)).shape))
print("   wide in the middle, d_model at both ends -- transformer.py:159")


Q (1, 3, 8)  K (1, 5, 16)  V (1, 5, 32)
-> (1, 3, 8)  back to embed_dim=8, because of out_proj: (8, 8)

So the module hides the freedom: inside, widths differ; at the door, embed_dim.
That is what makes the residual connection x + attn(x) legal.

DETR's FFN: (850, 1, 256) -> (850, 1, 2048) -> (850, 1, 256)
   wide in the middle, d_model at both ends -- transformer.py:159


## 4. Multi-head attention

One attention operation computes **one** kind of relationship. Real transformers run several in parallel — "heads" — and concatenate the results.

The trick: instead of `nheads` separate full-width attentions (expensive), **split** the existing width into `nheads` chunks. Same total compute, several independent relationships.

Each tensor is split along *its own* width, so with `D_k = 4` and `D_v = 6` the heads are 2-wide for queries and keys but 3-wide for values. Nothing requires those to agree — the rules from §3 still hold inside every head.

With the batch axis kept, splitting turns `(B, L, D)` into `(B, nheads, L, head_dim)`. `attention()` needs no change at all: it transposes the last two axes and softmaxes the last one, so it batches over *both* leading axes for free.

DETR uses `nheads=8` with `d_model=256`. All three widths are 256 there — but they are still `D_in`, `D_k` and `D_v` doing three different jobs, and each head gets 256/8 = 32 of each.

In [13]:
nheads = 2
print(f"D_k={D_k} -> {D_k // nheads} per head;   D_v={D_v} -> {D_v // nheads} per head")

def split_heads(t, nheads):
    B_, L_, D_ = t.shape
    return t.view(B_, L_, nheads, D_ // nheads).transpose(1, 2)   # (B, nheads, L, head_dim)

qh, kh, vh = split_heads(q, nheads), split_heads(k, nheads), split_heads(v, nheads)
print("after split: qh", tuple(qh.shape), " kh", tuple(kh.shape), " vh", tuple(vh.shape))
print("             = (B, heads, tokens, head_dim) -- qh/kh and vh differ in the last axis")

out_h, w_h = attention(qh, kh, vh)   # unchanged: it batches over B AND heads
print("per-head output:", tuple(out_h.shape), " weights:", tuple(w_h.shape))

merged = out_h.transpose(1, 2).reshape(B, L, D_v)    # concat the heads back together
print("after concat   :", tuple(merged.shape), "-- (B, L, D_v), what single-head attention gave")
print()
print("the two heads learned DIFFERENT attention patterns:")
print("head 0, token 0:", w_h[0, 0, 0])
print("head 1, token 0:", w_h[0, 1, 0])


D_k=4 -> 2 per head;   D_v=6 -> 3 per head
after split: qh (1, 2, 5, 2)  kh (1, 2, 5, 2)  vh (1, 2, 5, 3)
             = (B, heads, tokens, head_dim) -- qh/kh and vh differ in the last axis
per-head output: (1, 2, 5, 3)  weights: (1, 2, 5, 5)
after concat   : (1, 5, 6) -- (B, L, D_v), what single-head attention gave

the two heads learned DIFFERENT attention patterns:
head 0, token 0: tensor([0.316, 0.206, 0.129, 0.128, 0.221], grad_fn=<SelectBackward0>)
head 1, token 0: tensor([0.192, 0.220, 0.181, 0.236, 0.170], grad_fn=<SelectBackward0>)


### Verify against `nn.MultiheadAttention`

The real test: reimplement PyTorch's module by hand, pulling out its actual weights, and check the numbers match.

Two setup notes.

**`batch_first=True`.** PyTorch's default is `(L, B, E)` — *sequence* first, batch in the middle — and DETR keeps that default, which is why notebooks `03` onward are full of `(850, 1, 256)`. We pass `batch_first=True` here so the module speaks the `(B, L, …)` convention this notebook has used throughout. Getting this flag wrong is silent: with `B` and `L` both small, the module happily attends over the wrong axis and returns a plausibly-shaped tensor.

**One value, three widths.** `nn.MultiheadAttention` packs Wq, Wk and Wv into a **single** `in_proj_weight` of shape `(3E, E)`, and you cannot stack three matrices into one block unless they have the same shape. So all three widths take the value `embed_dim`. We still write `E_in`, `E_k`, `E_v` separately below — one number, three roles — because that is what makes `in_proj_weight`'s shape readable as *"an `(E_k, E_in)` block, an `(E_k, E_in)` block and an `(E_v, E_in)` block, stacked"* rather than an unexplained 24.

(For the general case the module takes `kdim` and `vdim`, as in §3; supply them and it builds three separate projections instead of one packed block.)

In [14]:
# One value, three roles. nn.MultiheadAttention requires them equal -- so we keep
# the names and let them share a number, rather than collapsing to a single `E`.
E_in = E_k = E_v = 8                   # == embed_dim
nheads = 2

mha = nn.MultiheadAttention(embed_dim=E_in, num_heads=nheads, bias=True, batch_first=True)
mha.eval()

seq = torch.randn(B, L, E_in)          # (B, L, E_in) -- our convention, thanks to batch_first
ref_out, ref_w = mha(seq, seq, seq)    # self-attention
print("module output:", tuple(ref_out.shape), " weights:", tuple(ref_w.shape))

# --- now do it by hand with mha's own weights ---
Wi, bi = mha.in_proj_weight, mha.in_proj_bias
print("in_proj_weight:", tuple(Wi.shape), f"= ({E_k} + {E_k} + {E_v}, {E_in})")
print("   three stacked blocks: (E_k, E_in), (E_k, E_in), (E_v, E_in)")
print("   they can only stack because the three widths happen to be equal.")

Wq_, Wk_, Wv_ = Wi.chunk(3, dim=0)                 # chunk(3): three pieces -- notebook 00 §2
bq_, bk_, bv_ = bi.chunk(3, dim=0)
print("   after chunk(3):", tuple(Wq_.shape), tuple(Wk_.shape), tuple(Wv_.shape))

# Wq_ is a 2-D weight matrix, so .T is safe and idiomatic here --
# unlike the 3-D activations above, which need transpose(-2, -1).
qm = seq @ Wq_.T + bq_                             # (B, L, E_k)
km = seq @ Wk_.T + bk_                             # (B, L, E_k)
vm = seq @ Wv_.T + bv_                             # (B, L, E_v)
om, wm = attention(split_heads(qm, nheads), split_heads(km, nheads), split_heads(vm, nheads))
om = om.transpose(1, 2).reshape(B, L, E_v)         # concat heads -> (B, L, E_v)
om = om @ mha.out_proj.weight.T + mha.out_proj.bias   # out_proj: E_v -> E_in, for the residual

print()
print("PyTorch output :", ref_out[0, 0])
print("my output      :", om[0, 0])
print()
print("outputs match           :", torch.allclose(ref_out, om, atol=1e-5))
print("attention weights match :", torch.allclose(ref_w, wm.mean(1), atol=1e-5))
print("   (nn.MultiheadAttention AVERAGES the heads' weights before returning them --")
print("    wm.mean(1) averages our head axis. That is why notebook 06's maps are")
print("    head-averaged unless you ask for average_attn_weights=False.)")


module output: (1, 5, 8)  weights: (1, 5, 5)
in_proj_weight: (24, 8) = (8 + 8 + 8, 8)
   three stacked blocks: (E_k, E_in), (E_k, E_in), (E_v, E_in)
   they can only stack because the three widths happen to be equal.
   after chunk(3): (8, 8) (8, 8) (8, 8)

PyTorch output : tensor([ 0.349,  0.342,  0.044, -0.098,  0.232, -0.086, -0.027, -0.119], grad_fn=<SelectBackward0>)
my output      : tensor([ 0.349,  0.342,  0.044, -0.098,  0.232, -0.086, -0.027, -0.119], grad_fn=<SelectBackward0>)

outputs match           : True
attention weights match : True
   (nn.MultiheadAttention AVERAGES the heads' weights before returning them --
    wm.mean(1) averages our head axis. That is why notebook 06's maps are
    head-averaged unless you ask for average_attn_weights=False.)


## 5. The punchline: attention is permutation-invariant

This is the property that shapes DETR's entire design. **Shuffle the input tokens and the outputs are just as shuffled — no information about order or position survives.**

Let's prove it.

In [17]:
perm = torch.randperm(L)                  # some shuffle of the L tokens
print("permutation:", perm.tolist())

o_orig, _ = attention(W_q(x), W_k(x), W_v(x))             # attention on the original order
xs = x[:, perm]                                           # shuffle tokens WITHIN each sequence
o_shuf, _ = attention(W_q(xs), W_k(xs), W_v(xs))          # attention on the shuffled order

print()
print("output of shuffled input == shuffled output of original input?")
print("  ", torch.allclose(o_shuf, o_orig[:, perm], atol=1e-6))
print()
print("Note x[:, perm], not x[perm] -- with a batch axis the tokens live on axis 1.")
print("x[perm] would shuffle SEQUENCES, and with B = 1 it would silently do nothing.")
print()
print("So attention has NO idea which token came first.")
print("To it, a sequence is an unordered BAG of tokens.")


permutation: [1, 2, 0, 3, 4]

output of shuffled input == shuffled output of original input?
   True

Note x[:, perm], not x[perm] -- with a batch axis the tokens live on axis 1.
x[perm] would shuffle SEQUENCES, and with B = 1 it would silently do nothing.

So attention has NO idea which token came first.
To it, a sequence is an unordered BAG of tokens.


### Two consequences, and they are the whole of notebooks `03` and `04`

**(a) You must inject position manually.** An image flattened into 850 tokens loses all 2-D layout. Detection is *entirely* about where things are — so DETR adds a **positional encoding** to every token. That is notebook `03` §5.

**(b) The N object queries must differ from one another.** If all 100 queries were identical vectors, permutation-invariance means they'd all produce **identical outputs** — 100 copies of the same box. The queries are *learned to be different*, which is what lets them specialize. That is notebook `04` §2.

Let's verify (b) directly, since it's the one people find surprising:

In [16]:
img_tokens = torch.randn(B, 6, D_in)                # (B, 6, D_in)

same = torch.randn(B, 1, D_in).expand(B, 3, D_in)   # 3 IDENTICAL queries
o_same, _ = attention(W_q(same), W_k(img_tokens), W_v(img_tokens))
print("3 identical queries -> are the 3 outputs identical?")
print("   out[0] == out[1]:", torch.allclose(o_same[0, 0], o_same[0, 1], atol=1e-6))
print("   -> 3 duplicate detections. Useless.\n")

diff = torch.randn(B, 3, D_in)                      # 3 DIFFERENT queries
o_diff, _ = attention(W_q(diff), W_k(img_tokens), W_v(img_tokens))
print("3 different queries -> outputs differ?")
print("   out[0] == out[1]:", torch.allclose(o_diff[0, 0], o_diff[0, 1], atol=1e-6))
print("   -> 3 distinct detections. This is why query_embed is LEARNED.")


3 identical queries -> are the 3 outputs identical?
   out[0] == out[1]: True
   -> 3 duplicate detections. Useless.

3 different queries -> outputs differ?
   out[0] == out[1]: False
   -> 3 distinct detections. This is why query_embed is LEARNED.


> *"Since the decoder is also permutation-invariant, the N input embeddings must be different to produce different results."* — paper §3.2

You have now derived that sentence yourself, rather than taking it on faith.

## 6. Exercises

Try each before opening the solution.

---

**Exercise 1.** Given `attn` of shape `(B, L, L)` from §2, what does `attn[0, 2].argmax()` tell you? What about `attn[0, :, 2].argmax()`?

<details><summary>Solution</summary>

`attn[0, 2].argmax()` — which token **token 2 attends to most** (row = one query's distribution, sums to 1).

`attn[0, :, 2].argmax()` — which token attends most **to token 2**. This is *not* a distribution and does not sum to 1. Rows and columns mean different things; mixing them up is the single most common attention bug.

```python
print("token 2 looks mostly at token", attn[0, 2].argmax().item())
print("token 2 is looked at most by token", attn[0, :, 2].argmax().item())
```
</details>

---

**Exercise 2.** In cross-attention with 5 queries and 900 image tokens, what shape is the attention weight matrix, and what shape is the output?

<details><summary>Solution</summary>

Weights `(B, 5, 900)`, output `(B, 5, D_v)`.

Output length always follows the **query**. This is exactly notebook `06`, where decoder cross-attention was `(100, 850)`: 100 queries × 850 image tokens.
</details>

---

**Exercise 3.** Remove the `/ √d` scaling from `attention()` and rerun §2 with `D_k = 512` instead of 4. What happens to the attention weights, and why does that stop the model learning?

<details><summary>Solution</summary>

The weights collapse to nearly one-hot (max ≈ 1.0). Softmax's gradient is `p(1-p)`, so when `p → 1` the gradient → 0 and no signal flows back. Scaling keeps scores in a range where softmax stays soft and differentiable.

```python
x2 = torch.randn(B, 4, 512)
Wq2 = nn.Linear(512, 512, bias=False)
s = Wq2(x2) @ Wq2(x2).transpose(-2, -1)     # unscaled
print("unscaled max weight:", s.softmax(-1).max().item())
print("  scaled max weight:", (s / math.sqrt(512)).softmax(-1).max().item())
```
</details>

---

**Exercise 4.** DETR's decoder adds the object query to Q and K, but **not** to V (notebook `03` §5). Using the dictionary analogy, why is that the right choice?

<details><summary>Solution</summary>

Q and K decide **where to look** — position should influence that. V is **what gets handed back** — that should be image content, not position. Mixing position into V would contaminate the retrieved features with coordinates.

The paper ablates this in Table 3: passing encodings into the attention layers beats adding them once at the input by 1.4 AP.
</details>

---

**Exercise 5 (harder).** Write `attention()` so it accepts a `key_padding_mask` of shape `(L,)` where `True` = padding to ignore. Verify masked positions get exactly zero weight.

<details><summary>Solution</summary>

Set masked scores to `-inf` **before** the softmax, so they become exactly 0 after it.

```python
def attention_masked(q, k, v, key_padding_mask=None):
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
    if key_padding_mask is not None:
        scores = scores.masked_fill(key_padding_mask, float("-inf"))
    w = scores.softmax(-1)
    return w @ v, w

mask = torch.tensor([False, False, True, True, True])   # ignore the last 3 tokens
_, w = attention_masked(q, k, v, mask)
print(w)
print("masked columns are exactly zero:", bool((w[..., 2:] == 0).all()))
```

This is precisely what DETR does with the padding masks from notebook `03` §1 — real images padded into a batch must not attend to the padding.
</details>

## You're ready

| You now know | Where DETR uses it |
|---|---|
| Q, K, V and scaled dot-product attention | everywhere |
| output length follows the **query** | 100 queries → 100 detections |
| self- vs cross-attention | encoder vs decoder |
| multi-head, and that returned weights are head-averaged | notebook `06`'s attention maps |
| attention is **permutation-invariant** | why positional encodings (`03`) and distinct queries (`04`) exist |
| padding masks via `-inf` before softmax | batching variable-sized images (`03`) |

Continue to **`02`** — why object detection needed a transformer in the first place.